# Milestone 2 lab, notebook 1 of 8: Ask only for what changed

## Where we are

| Notebook | What you do |
|---|---|
| **1. Ask only for what changed** (you are here) | Read the orders that changed, not all of them |
| 2. Remember where you stopped | Keep a watermark. See what happens when it moves too early |
| 3. When the API says no | Meet the rate limit and a server error. Wait, and try again |
| 4. Land it raw | Put every order in Postgres, exactly as it arrived |
| 5. Pull or push | A short lab: let the order system send the changes to you |
| 6. What you built has a name | Change data capture, and what this kind cannot see |
| 7. Counted twice | Run a few mornings. Watch the report go wrong |
| 8. The production version | The same logic as a tested program |

**Before you start,** open [`milestone_2_flow.html`](../milestone_2_flow.html) in a browser. It shows the whole pipeline at the end of this milestone.

Milestone 1 ended with a broken morning. The load read every order Arden ever took, and on 11 June the order system refused it. This notebook finds out how much of that reading was needed.

In [1]:
from lab_helpers import api, bars, check_api, money, read_orders, set_morning, show_doc

check_api()

The order system is running. Its clock says 2026-06-11 07:00:00 in Arden.


## Step 1: The cost of reading everything

It is the morning of Tuesday 9 June. Read every order, as milestone 1 did.

In [2]:
set_morning("2026-06-09")
everything, calls_full = read_orders()

print(f"Full load: {len(everything)} orders in {calls_full} calls.")

It is now Tuesday 2026-06-09 07:00:00 in Arden.
Full load: 251 orders in 11 calls.


The manager's report ran yesterday too, at 07:00 on Monday. Most of these orders were already in the warehouse then. How many are really different today?

## Step 2: The API can answer that

Every order has an `updated_at`: the last time anything about it changed. And the API has a filter for it.

In [3]:
show_doc("LLD.md", "2. The source: what is new in the API")

**From `docs/LLD.md`**

> ## 2. The source: what is new in the API
> 
> | Call | Returns |
> |---|---|
> | `GET /orders?updated_since=2026-06-08T03:48:53Z` | Only orders with `updated_at` **at or after** that time |
> | `POST /admin/faults` | Makes the next calls fail. Body: `{"kind": "error_500", "count": 3}` |
> | `POST /webhooks` | Lab only: registers an address that receives each change |
> 
> | Fault | What the next calls do |
> |---|---|
> | `error_500` | Answer 500, the server failed |
> | `rate_limit` | Answer 429 with `Retry-After: 1` |
> | `repeat_page` | Serve one page twice |
> 
> One fault is planted in the data: in every run of **2026-06-11**, the second page is served twice.

Yesterday's run was at 07:00 on Monday in Arden. In UTC that is `2026-06-08T12:00:00Z`. Ask only for orders changed at or after that moment.

In [4]:
set_morning("2026-06-09")      # the same morning again: this also gives a fresh allowance of API calls

monday_run = "2026-06-08T12:00:00Z"
changed, calls_changed = read_orders(updated_since=monday_run)

print(f"Changed since Monday's run: {len(changed)} orders in {calls_changed} calls.")

It is now Tuesday 2026-06-09 07:00:00 in Arden.
Changed since Monday's run: 29 orders in 2 calls.


In [5]:
bars({"Full load": len(everything), "Only what changed": len(changed)}, unit=" orders")
print()
bars({"Full load": calls_full, "Only what changed": calls_changed}, unit=" API calls")

Full load          ████████████████████████████████████████ 251 orders
Only what changed  █████ 29 orders

Full load          ████████████████████████████████████████ 11 API calls
Only what changed  ███████ 2 API calls


Nine tenths of the full load was the same as yesterday.

## Step 3: Is anything missing?

A smaller load is only good if it is still complete. Take the full load, and build Monday's warehouse and Tuesday's truth from it by hand. Then check that "Monday plus the changed orders" gives the same as "everything".

In [6]:
set_morning("2026-06-08")
monday, _ = read_orders()                     # what the warehouse held after Monday's run
set_morning("2026-06-09")

warehouse = {order["order_id"]: order for order in monday}
for order in changed:                         # put each changed order over the old one
    warehouse[order["order_id"]] = order

truth = {order["order_id"]: order for order in everything}
print("Monday's orders + the changed orders == a full load today?", warehouse == truth)

It is now Monday 2026-06-08 07:00:00 in Arden.
It is now Tuesday 2026-06-09 07:00:00 in Arden.
Monday's orders + the changed orders == a full load today? True


The same result, from 29 orders, not 251.

## Step 4: Two new words

| Word | Meaning | Here |
|---|---|---|
| **Incremental load** | "An incremental load reads only rows that are new or changed." | 29 orders on Tuesday |
| **Incremental key** | The column that says a row changed. "Pick a key that changes whenever the row changes." | `updated_at` |

The definitions are from the D8Loop chapters [Data Sources & Ingestion Patterns](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-sources-and-ingestion-patterns) and [Extraction: Pulling Data Reliably](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/extraction-pulling-data-reliably).

### Compare the key with a worse one

Why `updated_at`, and not `placed_at`? Try it. Keep the orders **placed** since Monday's run.

In [7]:
by_placed_at = [order for order in everything if order["placed_at"] >= monday_run]

print(f"Key = placed_at:  {len(by_placed_at)} orders")
print(f"Key = updated_at: {len(changed)} orders")

missed = [o for o in changed if o["placed_at"] < monday_run]
print(f"\nplaced_at misses {len(missed)} orders that changed. One of them:")
order = next(o for o in missed if o["refunded_cents"] > 0)
print(f"  order {order['order_id']}: placed {order['placed_at']}, refunded {money(order['refunded_cents'])} at {order['refunded_at']}")

Key = placed_at:  25 orders
Key = updated_at: 29 orders

placed_at misses 4 orders that changed. One of them:
  order 1130: placed 2026-06-05T15:20:00Z, refunded $5.78 at 2026-06-08T16:00:00Z


`placed_at` never changes, so it only finds **new** orders. It misses every order that was delivered, cancelled or refunded since yesterday. Order 1130's refund would never reach the warehouse, and Friday's number would stay wrong for ever.

## What you now know

- On Tuesday only 29 of 251 orders were new or changed.
- An incremental load reads only those. It gave the same result as the full load.
- The incremental key must change whenever the row changes. Here it is `updated_at`.

**Say it in an interview:** "I moved from a full load to an incremental load on `updated_at`. On one morning that cut the read from 251 orders to 29. I did not use the creation time as the key, because it would have missed every refund and cancellation."

But one thing was done by hand here: someone typed `2026-06-08T12:00:00Z`. [notebook 2](02_remember_where_you_stopped.ipynb) makes the pipeline remember that itself.